In [1]:
# ==============================================================================
# 📓 Advanced RAG Cookbook: 02_chunking.ipynb
# ==============================================================================
# الهدف: إتقان استراتيجيات تقطيع النصوص المختلفة من التقطيع الثابت وحتى 
# التقطيع الدلالي (Semantic Chunking) للحفاظ على السياق والمعنى.
# ==============================================================================

# تثبيت المكتبات المطلوبة للتشغيل المحتوي على Semantic Chunker
# !pip install langchain-text-splitters langchain-experimental langchain-community sentence-transformers

import os
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

print("✅ تم استيراد مكتبات التقطيع بنجاح!")

c:\Users\ENG_AMR\anaconda3\envs\enterprise_RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ تم استيراد مكتبات التقطيع بنجاح!


C:\Users\ENG_AMR\AppData\Local\Temp\ipykernel_14120\1953362566.py:13: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


<div dir="rtl">

## 1. التقطيع بفاصل ثابت (`CharacterTextSplitter`)

### 📌 الشرح النظري:
أبسط أنواع التقطيع، حيث يتم تقسيم النص بناءً على فاصل محدد مثل المسافة (`" "`) أو السطر الجديد (`"\n"`). يقوم بحساب عدد الحروف في كل قطعة حتى يصل للحد الأقصى المسموح به `chunk_size` ثم يقطع النص.

### 💡 المميزات:
* خفيف وسريع جداً في التنفيذ.
* مناسب للنصوص البسيطة والموحدة في التنسيق.

### ⚠️ العيوب:
* قد يقطع الجمل أو الأفكار من المنتصف بشكل عشوائي.
* يتجاهل الهيكل الطبيعي للغة كـ الفقرات والنقاط.

</div>

In [ ]:
# ==============================================================================
# 1. CharacterTextSplitter Demo
# ==============================================================================

# نص تجريبي للمعاينة
sample_text = """Retrieval-Augmented Generation (RAG) is an AI framework for improving the quality of LLM responses.
It anchors the model on external sources of knowledge to supplement the LLM's internal representation of information.
Implementing RAG in LLM-based systems enhances accuracy and reduces hallucinations."""

# 1. إعداد القاطع بحجم 100 حرف وتداخل 20 حرف
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=100,
    chunk_overlap=20,
    length_function=len
)

# 2. تطبيق التقطيع
char_chunks = char_splitter.split_text(sample_text)

# 3. طباعة النتائج
print("--- 1. CharacterTextSplitter Output ---")
print(f"🧩 إجمالي القطع (Chunks): {len(char_chunks)}\n")
for i, chunk in enumerate(char_chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

<div dir="rtl">

## 2. التقطيع الهيكلي التكراري (`RecursiveCharacterTextSplitter`)

### 📌 الشرح النظري:
الطريقة المعيارية الشائعة لتقطيع النصوص. بدلاً من القطع بناءً على رمز واحد، تستخدم قائمة من الفواصل مرتبة حسب الأولوية:
1. الفقرات: `"\n\n"`
2. الأسطر الجديدة: `"\n"`
3. المسافات: `" "`
4. الحروف الفردية: `""`

تحاول هذه الطريقة إبقاء الفقرة كاملة معاً، وإذا كانت أطول من `chunk_size` تنتقل للفاصل الأدنى كالأسطر أو الكلمات.

### 💡 المميزات:
* تحافظ على المفهوم العام للفقرة والجمل قدر الإمكان.
* خيار قياسي ممتاز لمعظم أنواع المستندات العامة.

### ⚠️ العيوب:
* لا تفهم معنى النص بشكل دلالي، بل تعتمد على الفواصل الشكلية فقط.

</div>

In [ ]:
# ==============================================================================
# 2. RecursiveCharacterTextSplitter Demo
# ==============================================================================

# 1. إعداد القاطع التكراري بحجم 150 حرف وتداخل 30 حرف
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", " ", ""]
)

# 2. تطبيق التقطيع على نفس النص
recursive_chunks = recursive_splitter.split_text(sample_text)

# 3. طباعة النتائج
print("--- 2. RecursiveCharacterTextSplitter Output ---")
print(f"🧩 إجمالي القطع (Chunks): {len(recursive_chunks)}\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

<div dir="rtl">

## 3. التقطيع الدلالي (`SemanticChunker`)

### 📌 الشرح النظري:
أحدث وأقوى استراتيجيات التقطيع في الـ Advanced RAG. تعمل بالخطوات التالية:
1. تقسم النص بالكامل إلى جمل منفصلة.
2. تحسب الـ Embeddings لكل جملة والجملة التي تليها وتحدد درجة التشابه الدلالي (`Cosine Similarity`).
3. عند حدوث هبوط حاد في نسبة التشابه بين جملتين (انتقال من موضوع لموضوع آخر)، تقوم بعمل قطع وتشكيل `Chunk` جديد.

أنواع عتبة القطع (`breakpoint_threshold_type`):
* `percentile`: يقطع عند الفروقات النسبية في المعنى وهو الشائع والمفضل.
* `standard_deviation`: يقطع بناءً على الانحراف المعياري للتشابه.

### 💡 المميزات:
* تضمن احتواء كل `Chunk` على فكرة واحدة مكتملة ودقيقة.
* تقلل بشكل كبير من مشكلة فقدان السياق والـ Hallucinations عند الاسترجاع.

### ⚠️ العيوب:
* أبطأ من الطرق العادية لأنها تتطلب نموذج Embeddings لتقييم كل جملة.

</div>

In [ ]:
# ==============================================================================
# 3. SemanticChunker Demo
# ==============================================================================

# 1. إعداد نموذج Embeddings محلي وخفيف للحسابات الدلالية
print("⏳ جاري تحميل نموذج الـ Embeddings للتقطيع الدلالي...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. إعداد الـ Semantic Chunker باستخدام Percentile
semantic_splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90  # درجة الحساسية لتغير المعنى
)

# 3. نص تجريبي ينتقل بين موضوعين مختلفين تماماً (الفضاء والطبخ)
multi_topic_text = """The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy.
As the largest optical telescope in space, its high resolution and sensitivity allow it to view objects too old or distant.
It was launched in December 2021 by NASA.

Pasta is a staple food of traditional Italian cuisine, with its first attested documentations dating to 1154 in Sicily.
It is typically made from an unleavened dough of wheat flour mixed with water or eggs and formed into sheets.
Spaghetti and penne are popular types of dry pasta."""

# 4. تطبيق التقطيع الدلالي
semantic_docs = semantic_splitter.create_documents([multi_topic_text])

# 5. طباعة النتائج
print("\n--- 3. SemanticChunker Output ---")
print(f"🧩 إجمالي القطع الدلالية: {len(semantic_docs)}\n")
for i, doc in enumerate(semantic_docs):
    print(f"--- Semantic Chunk {i+1} ---")
    print(doc.page_content.strip())
    print()

<div dir="rtl">

## 📝 الخلاصة للـ Notebook (Takeaways)

1. **التقطيع التقليدي (`RecursiveCharacterTextSplitter`):** ممتاز كمرحلة أولى، وسريع للبيانات الضخمة أو المشاريع ذات الميزانية المحدودة.
2. **التقطيع الدلالي (`SemanticChunker`):** هو الخيار الأفضل للأوراق البحثية والتقارير المعقدة التي تتطلب حماية سياق المعنى بدقة.
3. **تداخل النصوص (`chunk_overlap`):** أضف دائماً تداخل بنسبة 10 إلى 20 بالمئة عند استخدام الطرق التقليدية لمنع ضياع الكلمات الموجودة على حدود القطع.

</div>